# OpenFold with Real Protein Sequences (UniRef50)

Tests OpenFold Evoformer on **real protein sequences** from UniRef50,
**binned by sequence length** to match the ProtMamba UniRef experiment.

## Purpose

- Tests ecological validity: do real proteins show the same geometric behavior as synthetic?
- Length-binned evaluation enables direct comparison with ProtMamba UniRef results
- Pipeline: `AA sequence → ESM-1b → OpenFold Evoformer → Single Repr (c_s=384) → Mean Pool → StabilityHarness`

## Setup

1. Upload `evaluation_harness.py` and `perturbation_protocol.py`
2. Change Runtime to GPU (A100 recommended)
3. Run cells in order

---

In [ ]:
OPENFOLD_DIR = os.environ.get('OPENFOLD_DIR', '/content/openfold')
WEIGHTS_DIR = os.environ.get('OF_WEIGHTS_DIR', './results/openfold_weights')

# Install Dependencies
import os, sys

print('Installing base dependencies...')
!pip install -q torch matplotlib seaborn pandas scipy scikit-learn

print('Installing ESM (Facebook)...')
!pip install -q fair-esm

print('Installing shesha-geometry...')
!pip install -q shesha-geometry

!pip install -q datasets biopython ml-collections modelcif awscli

print('Installing OpenFold...')
try:
    import openfold
    print(f'  OpenFold already installed: {openfold.__file__}')
except ImportError:
    !pip install -q "openfold @ git+https://github.com/aqlaboratory/openfold.git" 2>/dev/null
    try:
        import openfold
        print(f'  OpenFold installed via pip: {openfold.__file__}')
    except ImportError:
        if not os.path.exists(OPENFOLD_DIR):
            print('  Cloning OpenFold repo...')
            !git clone --depth 1 -q https://github.com/aqlaboratory/openfold.git {OPENFOLD_DIR}
        sys.path.insert(0, OPENFOLD_DIR)
        try:
            import openfold
            print(f'  OpenFold loaded from clone: {openfold.__file__}')
        except ImportError as e:
            print(f'  ERROR: Could not install OpenFold: {e}')

# Download SoloSeq weights
WEIGHTS_DIR = WEIGHTS_DIR
WEIGHTS_FILE = 'seq_model_esm1b_ptm.pt'
WEIGHTS_PATH = f'{WEIGHTS_DIR}/{WEIGHTS_FILE}'
MIN_WEIGHTS_SIZE_BYTES = 50_000_000

def _ensure_soloseq_weights():
    global WEIGHTS_PATH
    if os.path.exists(WEIGHTS_PATH):
        size = os.path.getsize(WEIGHTS_PATH)
        if size >= MIN_WEIGHTS_SIZE_BYTES:
            return True
        print(f"Removing incomplete weights ({size / 1e6:.0f} MB)...")
        os.remove(WEIGHTS_PATH)
    os.makedirs(WEIGHTS_DIR, exist_ok=True)
    print("\nDownloading SoloSeq weights via AWS S3...")
    import subprocess, shutil
    # Direct S3 download (no credentials needed)
    dl_dir = f'{WEIGHTS_DIR}/openfold_soloseq_params'
    os.makedirs(dl_dir, exist_ok=True)
    subprocess.run([
        'aws', 's3', 'cp', '--no-sign-request', '--region', 'us-east-1',
        's3://openfold/openfold_soloseq_params/', dl_dir, '--recursive'
    ], check=False)
    # Move weights file to expected location
    for root, _, files in os.walk(WEIGHTS_DIR):
        for f in files:
            if f == WEIGHTS_FILE:
                src = os.path.join(root, f)
                if os.path.abspath(src) != os.path.abspath(WEIGHTS_PATH):
                    print(f"  Copying {src} -> {WEIGHTS_PATH}")
                    shutil.copy2(src, WEIGHTS_PATH)
                break
    return os.path.exists(WEIGHTS_PATH) and os.path.getsize(WEIGHTS_PATH) >= MIN_WEIGHTS_SIZE_BYTES

if _ensure_soloseq_weights():
    print(f'\nSoloSeq weights ready: {WEIGHTS_PATH} ({os.path.getsize(WEIGHTS_PATH)/1e6:.0f} MB)')
else:
    print(f'\nWARNING: Weights not found or incomplete at {WEIGHTS_PATH}')

import numpy as np
import torch
print(f'torch {torch.__version__} | CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print('Done!')

In [ ]:
# Configuration

PHASE = 'full'

# Length bins for analysis: (min, max) inclusive
LENGTH_BINS = [
    (50, 150),
    (151, 250),
    (251, 350),
    (351, 450),
    (451, 550),
    (551, 650),
    (651, 800),
]

OUTPUT_BASE = './results/openfold_uniref_experiment/'
RESULTS_DIR = OUTPUT_BASE + 'results'
CACHE_DIR = OUTPUT_BASE + 'cache'

CONFIG = {
    'quick': {'n_sequences': 500, 'min_length': 50, 'max_length': 800,
        'models': ['OpenFold_SoloSeq'], 'batch_size': 1, 'n_bootstrap': 0,
        'snp_rates': [0.01, 0.05, 0.10]},
    'full': {'n_sequences': 400000, 'min_length': 50, 'max_length': 800,
        'models': ['OpenFold_SoloSeq'], 'batch_size': 1, 'n_bootstrap': 5,
        'snp_rates': [0.01, 0.02, 0.05, 0.10]},
}
config = CONFIG[PHASE]

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# Set to True on first run to ONLY cache embeddings (skip Shesha analysis).
# Set to False on a later run to compute stability from cached embeddings.
EMBEDDINGS_ONLY = True

print(f'Phase: {PHASE.upper()} | Data: UniRef50 (real proteins)')
print(f'Length bins: {LENGTH_BINS}')
print(f'Sequences to scan: {config["n_sequences"]}')

In [ ]:
# Load Real Protein Sequences from UniProt
#
# Downloads real protein sequences directly from UniProt (Swiss-Prot reviewed).
# ~570K high-quality, manually curated protein sequences.
# Much more reliable than HuggingFace UniRef50 samples.

import os
import numpy as np
import urllib.request
import time

VALID_AAS = set('ACDEFGHIKLMNPQRSTVWY')

def download_uniprot_sequences(n_sequences, min_len, max_len, cache_dir, seed=320):
    """Download real protein sequences from UniProt Swiss-Prot."""
    cache_file = f'{cache_dir}/uniprot_swissprot_{min_len}_{max_len}.txt'
    if os.path.exists(cache_file):
        print(f'Loading cached sequences from {cache_file}')
        with open(cache_file) as f:
            sequences = [line.strip() for line in f if line.strip()]
        print(f'  Loaded {len(sequences)} sequences')
        return sequences

    print('Downloading from UniProt Swiss-Prot (reviewed proteins)...')
    print('  This may take 2-5 minutes for ~570K proteins...')

    # Swiss-Prot: all reviewed proteins (high quality, ~570K sequences)
    url = 'https://rest.uniprot.org/uniprotkb/stream?format=fasta&query=(reviewed:true)&size=500000'
    req = urllib.request.Request(url, headers={'User-Agent': 'Python/alphagenome-experiment'})

    t0 = time.time()
    with urllib.request.urlopen(req, timeout=600) as response:
        raw = response.read().decode('ascii', errors='ignore')
    print(f'  Downloaded {len(raw)/1e6:.1f} MB in {time.time()-t0:.0f}s')

    # Parse FASTA
    proteins = []
    current_seq = []
    for line in raw.split('\n'):
        if line.startswith('>'):
            if current_seq:
                proteins.append(''.join(current_seq))
            current_seq = []
        elif line.strip():
            current_seq.append(line.strip().upper())
    if current_seq:
        proteins.append(''.join(current_seq))
    print(f'  Parsed {len(proteins)} protein sequences')

    # Filter by length, clean non-standard AAs
    sequences = []
    for p in proteins:
        if min_len <= len(p) <= max_len:
            cleaned = ''.join(c if c in VALID_AAS else 'A' for c in p)
            sequences.append(cleaned)

    print(f'  After length filter [{min_len}-{max_len}]: {len(sequences)} sequences')

    # Subsample if we got more than needed
    rng = np.random.default_rng(seed)
    if len(sequences) > n_sequences:
        idx = rng.choice(len(sequences), n_sequences, replace=False)
        sequences = [sequences[i] for i in sorted(idx)]
        print(f'  Subsampled to {len(sequences)} sequences')

    # Cache
    os.makedirs(cache_dir, exist_ok=True)
    with open(cache_file, 'w') as f:
        f.write('\n'.join(sequences))
    print(f'  Cached to {cache_file}')
    return sequences

# Load sequences
all_sequences = download_uniprot_sequences(
    n_sequences=config['n_sequences'],
    min_len=config['min_length'],
    max_len=config['max_length'],
    cache_dir=CACHE_DIR,
)

# Bin sequences by length
binned_sequences = {}
for lo, hi in LENGTH_BINS:
    binned = [s for s in all_sequences if lo <= len(s) < hi]
    if binned:
        binned_sequences[(lo, hi)] = binned
        print(f'  Bin [{lo}-{hi}): {len(binned)} sequences')
    else:
        print(f'  Bin [{lo}-{hi}): EMPTY (skipping)')

print(f'\nTotal sequences: {len(all_sequences)}')
print(f'Active bins: {len(binned_sequences)}')


In [ ]:
# Protein Perturbation Suite

from dataclasses import dataclass, field
AMINO_ACIDS = list('ACDEFGHIKLMNPQRSTVWY')

@dataclass
class PerturbedSet:
    name: str; category: str; sequences: list
    params: dict = field(default_factory=dict); description: str = ''

def mutate_protein(sequence, mutation_rate, rng):
    seq = list(sequence); n = max(1, int(len(seq) * mutation_rate))
    for pos in rng.choice(len(seq), size=n, replace=False):
        seq[pos] = rng.choice([aa for aa in AMINO_ACIDS if aa != seq[pos]])
    return ''.join(seq)

class ProteinPerturbationSuite:
    def __init__(self, seed=320, snp_rates=None, include_reverse=True):
        self.rng = np.random.default_rng(seed)
        self.snp_rates = snp_rates or [0.01, 0.02, 0.05, 0.10]
        self.include_reverse = include_reverse
    def run_all(self, sequences):
        results = {}
        for rate in self.snp_rates:
            name = f"aa_sub_{int(rate*100)}pct"
            results[name] = PerturbedSet(name=name, category='substitution',
                sequences=[mutate_protein(s, rate, self.rng) for s in sequences],
                params={'mutation_rate': rate})
        if self.include_reverse:
            results['reverse'] = PerturbedSet(name='reverse', category='reverse',
                sequences=[s[::-1] for s in sequences])
        return results

suite = ProteinPerturbationSuite(seed=320, snp_rates=config['snp_rates'])
print('Perturbation suite ready')

In [ ]:
# OpenFold Embedding Function (ESM-1b + SoloSeq Evoformer)

import torch
import torch.nn.functional as F
import gc

RESTYPES = 'ARNDCQEGHILKMFPSTWYV'
AA_TO_IDX = {aa: i for i, aa in enumerate(RESTYPES)}

def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache(); torch.cuda.synchronize()

def make_openfold_embedding_fn(device='cuda', max_length=1022):
    import esm as esm_module
    from openfold.config import model_config
    from openfold.model.model import AlphaFold

    print('Loading ESM-1b...')
    esm_model, esm_alphabet = esm_module.pretrained.esm1b_t33_650M_UR50S()
    esm_model = esm_model.eval().to(device)
    batch_converter = esm_alphabet.get_batch_converter()
    ESM_REPR_LAYER = 33
    esm_params = sum(p.numel() for p in esm_model.parameters()) / 1e6
    print(f'  ESM-1b loaded ({esm_params:.0f}M params)')

    print('Loading OpenFold SoloSeq...')
    cfg = model_config('seq_model_esm1b_ptm', train=False)
    cfg.model.template.enabled = False
    cfg.globals.use_flash = False
    cfg.globals.use_deepspeed_evo_attention = False
    of_model = AlphaFold(cfg)
    state = torch.load(WEIGHTS_PATH, map_location='cpu')
    of_model.load_state_dict(state, strict=False)
    of_model = of_model.eval().to(device)
    of_params = sum(p.numel() for p in of_model.parameters()) / 1e6
    print(f'  OpenFold loaded ({of_params:.0f}M params)')

    _captured = {}
    def _evo_hook(module, inp, out):
        m, z, s = out
        _captured['s'] = s.detach()
    of_model.evoformer.register_forward_hook(_evo_hook)

    @torch.no_grad()
    def embed(sequences):
        all_emb = []
        n = len(sequences)
        for idx, seq in enumerate(sequences):
            seq = seq[:max_length]
            L = len(seq)
            data = [('protein', seq)]
            _, _, tokens = batch_converter(data)
            tokens = tokens.to(device)
            esm_out = esm_model(tokens, repr_layers=[ESM_REPR_LAYER])
            esm_repr = esm_out['representations'][ESM_REPR_LAYER]
            seq_embedding = esm_repr[0, 1:L+1, :]
            aatype = torch.tensor([AA_TO_IDX.get(a, 20) for a in seq], dtype=torch.long, device=device)
            target_feat = F.one_hot(aatype.clamp(max=21), num_classes=22).float()
            residue_index = torch.arange(L, dtype=torch.long, device=device)
            seq_mask = torch.ones(L, device=device, dtype=torch.float32)
            msa_mask = torch.ones(1, L, device=device, dtype=torch.float32)
            NUM_ITERS = 1
            features = {
                'aatype': aatype.unsqueeze(-1).expand(-1, NUM_ITERS),
                'target_feat': target_feat.unsqueeze(-1).expand(-1, -1, NUM_ITERS),
                'residue_index': residue_index.unsqueeze(-1).expand(-1, NUM_ITERS),
                'seq_embedding': seq_embedding.unsqueeze(-1).expand(-1, -1, NUM_ITERS),
                'seq_mask': seq_mask.unsqueeze(-1).expand(-1, NUM_ITERS),
                'msa_mask': msa_mask.unsqueeze(-1).expand(-1, -1, NUM_ITERS),
            }
            _captured.clear()
            try:
                _ = of_model(features)
            except Exception:
                pass
            if 's' in _captured:
                s = _captured['s'].squeeze(0)
                pooled = s.mean(dim=0).cpu().numpy()
            else:
                pooled = seq_embedding.mean(dim=0).cpu().numpy()
            all_emb.append(pooled)
            if (idx + 1) % 10 == 0 or idx == n - 1:
                print(f'    Sequence {idx+1}/{n}', end='\r')
        print()
        return np.stack(all_emb)

    return embed, of_model, of_params

print('OpenFold embedding wrapper ready')

In [ ]:
# Patch evaluation_harness.py for compatibility
if os.path.exists('evaluation_harness.py'):
    with open('evaluation_harness.py', 'r') as _f:
        _src = _f.read()
    _changed = False
    if 'Literal' in _src and 'import Literal' not in _src and 'from __future__' not in _src:
        _src = _src.replace(
            'from typing import Optional',
            'from typing import Optional\ntry:\n    from typing import Literal\nexcept ImportError:\n    Literal = str'
        )
        _changed = True
    if 'from __future__ import annotations' not in _src:
        _src = 'from __future__ import annotations\n' + _src
        _changed = True
    if _changed:
        with open('evaluation_harness.py', 'w') as _f:
            _f.write(_src)
        print('Patched evaluation_harness.py')
    else:
        print('evaluation_harness.py already OK')
else:
    print('evaluation_harness.py not found in working directory')

In [ ]:
# Evaluation Harness
from evaluation_harness import StabilityHarness
harness = StabilityHarness(window_size=0, metric='cosine', n_splits=30,
                           seed=320, max_samples=2500, n_bootstrap=config['n_bootstrap'])
print('Harness configured')

In [ ]:
# Run Experiment (Length-Binned Evaluation)

import time
import pandas as pd
from evaluation_harness import ModelReport

device = 'cuda' if torch.cuda.is_available() else 'cpu'

all_detailed_rows = []

for (lo, hi), sequences in binned_sequences.items():
    bin_label = f'{lo}-{hi}'
    print(f"\n{'#'*70}")
    print(f"LENGTH BIN: [{bin_label}) — {len(sequences)} sequences")
    print(f"{'#'*70}")

    # Build perturbation suite for this bin
    suite = ProteinPerturbationSuite(snp_rates=config['snp_rates'])

    for model_idx, model_name in enumerate(config['models']):
        print(f"\n{'='*70}")
        print(f"[{model_idx+1}/{len(config['models'])}] {model_name} (bin={bin_label})")
        print('=' * 70)

        start_time = time.time()
        embed_fn, model_obj, n_params_m = make_openfold_embedding_fn(device=device)

        perturbed_sets = suite.run_all(sequences)

        safe_name = f'openfold_uniref_bin{lo}_{hi}'
        cache_clean = f'{CACHE_DIR}/{safe_name}_clean.npy'

        if os.path.exists(cache_clean):
            print(f'Loading cached: {cache_clean}')
            embeddings_clean = np.load(cache_clean)
            embeddings_clean = np.nan_to_num(embeddings_clean, nan=0.0, posinf=1e6, neginf=-1e6)
        else:
            print(f'Computing clean embeddings ({len(sequences)} seqs, bin=[{bin_label}))...')
            embeddings_clean = embed_fn(sequences)
            np.save(cache_clean, embeddings_clean)
            embeddings_clean = np.nan_to_num(embeddings_clean, nan=0.0, posinf=1e6, neginf=-1e6)
        print(f'  Shape: {embeddings_clean.shape}')

        perturbed_embeddings = {}
        for pert_name, pset in perturbed_sets.items():
            cache_pert = f'{CACHE_DIR}/{safe_name}_{pert_name}.npy'
            if os.path.exists(cache_pert):
                perturbed_embeddings[pert_name] = np.load(cache_pert)
                perturbed_embeddings[pert_name] = np.nan_to_num(perturbed_embeddings[pert_name], nan=0.0, posinf=1e6, neginf=-1e6)
            else:
                print(f'  Embedding: {pert_name}...')
                perturbed_embeddings[pert_name] = embed_fn(pset.sequences)
                np.save(cache_pert, perturbed_embeddings[pert_name])
                perturbed_embeddings[pert_name] = np.nan_to_num(perturbed_embeddings[pert_name], nan=0.0, posinf=1e6, neginf=-1e6)

        del model_obj
        cleanup_gpu()

        if EMBEDDINGS_ONLY:
            elapsed = time.time() - start_time
            print(f'\nBin [{bin_label}) embeddings cached in {elapsed/60:.1f} min (Shesha SKIPPED)')
        else:
            print('\nRunning Shesha evaluation...')
            report = harness.evaluate_all_perturbations(
                model_name=model_name, embeddings_clean=embeddings_clean,
                perturbed_dict=perturbed_embeddings)

            for pert_name, metrics in report.perturbation_breakdown().items():
                all_detailed_rows.append({
                    'model': model_name, 'size_M': round(n_params_m, 1),
                    'length_bin': bin_label,
                    'bin_lo': lo, 'bin_hi': hi,
                    'n_sequences': len(sequences),
                    'perturbation': pert_name,
                    'rdm_similarity': metrics.get('rdm_similarity_score', 0),
                    'rdm_drift': metrics.get('rdm_drift', 0),
                    'pert_stability': metrics.get('perturbation_stability_score', 0),
                    'pert_magnitude': metrics.get('perturbation_magnitude', 0),
                    'composite': metrics.get('composite_stability', 0),
                })

            elapsed = time.time() - start_time
            summary = report.summary()
            print(f'\nBin [{bin_label}) completed in {elapsed/60:.1f} min')
            print(f'  Composite: {summary["mean_composite_stability"]:.4f}')

# Save combined results
if EMBEDDINGS_ONLY:
    print('\n' + '=' * 70)
    print('EMBEDDINGS_ONLY mode: All embeddings cached. Set EMBEDDINGS_ONLY = False and re-run for Shesha analysis.')
    print('=' * 70)
df_detailed = pd.DataFrame(all_detailed_rows)
detailed_path = f'{RESULTS_DIR}/openfold_uniref_binned_{PHASE}_detailed.csv'
df_detailed.to_csv(detailed_path, index=False)
print(f'\nSaved to {detailed_path}')
print('\n' + '=' * 70)
print('LENGTH-BINNED EXPERIMENT COMPLETE')
print('=' * 70)

In [ ]:
# Visualization — Length-Binned Results

import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: composite by length bin, one line per perturbation
ax = axes[0]
for pert in df_detailed['perturbation'].unique():
    subset = df_detailed[df_detailed['perturbation'] == pert]
    agg = subset.groupby('bin_lo')['composite'].mean().reset_index().sort_values('bin_lo')
    ax.plot(agg['bin_lo'] + 25, agg['composite'], 'o-', label=pert, linewidth=2)
ax.set_xlabel('Sequence Length Bin (center)', fontsize=12)
ax.set_ylabel('Composite Stability', fontsize=12)
ax.set_title('OpenFold (UniRef50): Stability by Sequence Length', fontsize=14)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_ylim(0, 1.05)

# Right: mean composite vs length bin
ax = axes[1]
agg_mean = df_detailed.groupby('bin_lo')['composite'].mean().reset_index().sort_values('bin_lo')
counts = df_detailed.groupby('bin_lo')['n_sequences'].first().reset_index().sort_values('bin_lo')
ax.bar(agg_mean['bin_lo'] + 25, agg_mean['composite'], width=40, alpha=0.7, color='steelblue', edgecolor='navy')
ax.set_xlabel('Sequence Length Bin (center)', fontsize=12)
ax.set_ylabel('Mean Composite Stability', fontsize=12)
ax.set_title('OpenFold (UniRef50): Mean Stability by Length', fontsize=14)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim(0, 1.05)
for _, row in counts.iterrows():
    ax.text(row['bin_lo'] + 25, 0.02, f'n={int(row["n_sequences"])}', ha='center', fontsize=8, color='white', fontweight='bold')

plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/openfold_uniref_length_bins.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved plot to {RESULTS_DIR}/openfold_uniref_length_bins.png')